# OpenAI Agents SDK — the basics

The OpenAI Agents SDK is a thin layer over the model API: an `Agent` is a model plus
instructions plus tools, and a `Runner` runs the loop until the agent is done.
This notebook walks through the parts you need for almost any agent system:

1. **Agents and the agent loop** — the smallest thing that works
2. **Tracing and streaming** — seeing what happened, and showing tokens as they arrive
3. **Tools** — letting an agent do something, not just say something
4. **Memory** — carrying a conversation across separate runs
5. **Orchestrating by code** — deterministic control flow you write yourself
6. **Orchestrating by LLM** — agents as tools, and handoffs
7. **Structured outputs** — getting a Python object back instead of prose
8. **Guardrails** — blocking output you do not want to ship

The running example is a small writing team: several agents draft a short LinkedIn
post in different voices, an editor picks the best one, and a tool saves it.

Docs: <https://openai.github.io/openai-agents-python/>

**Setup.** The package on PyPI is `openai-agents` (installing `agents` gets an unrelated
library), and it is already a dependency of this repo. You need an `OPENAI_API_KEY` in
the `.env` file at the repo root.

In [ ]:
import asyncio
import os

from agents import (
    Agent,
    GuardrailFunctionOutput,
    ModelSettings,
    RunContextWrapper,
    Runner,
    SQLiteSession,
    function_tool,
    output_guardrail,
    trace,
)
from agents.exceptions import OutputGuardrailTripwireTriggered
from dotenv import find_dotenv, load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from pydantic import BaseModel, Field

load_dotenv(find_dotenv(usecwd=True), override=True)

MODEL = "gpt-5.4-mini"

if os.getenv("OPENAI_API_KEY"):
    print("OpenAI API key loaded")
else:
    print("OPENAI_API_KEY is not set — add it to .env before running the cells below")

## 1. Agents and the agent loop

An `Agent` is a name, a set of instructions (the system prompt) and a model.
`Runner.run()` sends the input to the model, executes any tool calls the model asks
for, feeds the results back, and repeats until the model returns a final answer.
For a plain agent with no tools, that loop is a single model call.

`Runner.run()` is async, so it is awaited directly in a notebook cell.

In [ ]:
writer = Agent(
    name="Post Writer",
    instructions="You write short, plain LinkedIn posts about AI engineering. No emoji, no hashtags.",
    model=MODEL,
)

result = await Runner.run(writer, "Write a post about why agent frameworks are so lightweight.")
print(result.final_output)

The result carries more than the text. `to_input_list()` returns the full conversation
as a list of message dicts — the same shape you would pass back in as input, which is
what makes the memory patterns in section 4 possible.

In [ ]:
result.to_input_list()

## 2. Tracing and streaming

### Tracing

Everything inside a `trace()` block is grouped into one workflow you can inspect at
<https://platform.openai.com/traces>: each model call, each tool call, each handoff,
with inputs, outputs and timings. This is the fastest way to debug agent behaviour,
and it costs nothing to switch on.

In [ ]:
with trace("Single post"):
    result = await Runner.run(writer, "Write a post about what tracing tells you about an agent run.")

print(result.final_output)

### Streaming

`Runner.run_streamed()` returns immediately and yields events as they happen. Most of
those events are about the run itself (a tool starting, an agent switching); the raw
token deltas arrive as `ResponseTextDeltaEvent`, which is what you print to make the
answer type itself out.

In [ ]:
result = Runner.run_streamed(writer, input="Write three one-line posts about AI agents.")

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

## 3. Tools

A tool is a Python function the model is allowed to call. The `@function_tool`
decorator reads the signature, type hints and docstring and generates the JSON schema
the model sees — so the docstring is not documentation, it is the prompt that tells
the model when and how to use the function.

The tool here appends to a list so you can verify the agent really called it. In a
real system it would write to a database or call an API; nothing else changes.

In [ ]:
saved_drafts: list[dict[str, str]] = []


@function_tool
def save_draft(title: str, body: str) -> str:
    """Save a post draft so it can be reviewed and published later.

    Args:
        title: A short title for the draft.
        body: The full text of the post.

    Returns:
        A confirmation message including how many drafts are now saved.
    """
    saved_drafts.append({"title": title, "body": body})
    return f"Draft saved: {title} ({len(saved_drafts)} saved in total)"

This is what the model actually receives — the description comes from the docstring,
the parameters from the type hints.

In [ ]:
print(save_draft.name)
print(save_draft.description)
save_draft.params_json_schema

In [ ]:
drafter = Agent(
    name="Drafter",
    instructions="You write short LinkedIn posts and save them with your tool.",
    model=MODEL,
    tools=[save_draft],
)

with trace("Draft and save"):
    result = await Runner.run(drafter, "Write a post about why tools are the interesting part of an agent, then save it.")

print(result.final_output)
saved_drafts

## 4. Memory

Within a single `Runner.run()` the conversation history is maintained across tool calls
and turns of the loop. But each new `Runner.run()` starts from nothing — the agent has
no memory of the previous call.

In [ ]:
assistant = Agent(name="Assistant", model=MODEL)

response = await Runner.run(assistant, "Hi there. My name is George.")
print(response.final_output)

response = await Runner.run(assistant, "What is my name?")
print(response.final_output)

### Approach 1: pass the history back yourself

`to_input_list()` gives you the conversation so far. Append the next user message and
pass the whole list as the input. Explicit, framework-independent, and you control
exactly what is remembered.

In [ ]:
first = await Runner.run(assistant, "Hi there. My name is George.")

next_input = first.to_input_list() + [{"role": "user", "content": "What is my name?"}]
second = await Runner.run(assistant, next_input)

print(second.final_output)

### Approach 2: a session

`SQLiteSession` does the same bookkeeping for you. Pass the same session to several
runs and the history is loaded and saved automatically. Given a filename it persists to
disk (`SQLiteSession("george", "memory.db")`), which survives a restart; without one it
lives in memory only. The first argument is the conversation id, so one database can
hold many separate conversations.

In [ ]:
session = SQLiteSession("george")

await Runner.run(assistant, "Hi there. My name is George.", session=session)
response = await Runner.run(assistant, "What is my name?", session=session)

print(response.final_output)

## 5. Orchestrating by code

There are two ways to run several agents together. **By code**: you write the control
flow in Python, so the sequence is deterministic and testable. **By LLM**: a model
decides which agent to call next, which is more flexible and less predictable.

Start with code. Three writers share a brief but differ in voice.

In [ ]:
BRIEF = """You write LinkedIn posts for an AI engineer sharing lessons from building
production agent systems. Keep posts under 120 words. No emoji, no hashtags."""

professional = Agent(
    name="Professional Writer",
    instructions=BRIEF + "\nYour voice is measured, precise and credible.",
    model=MODEL,
)
witty = Agent(
    name="Witty Writer",
    instructions=BRIEF + "\nYour voice is light and funny, but never silly.",
    model=MODEL,
)
concise = Agent(
    name="Concise Writer",
    instructions=BRIEF + "\nYour voice is blunt and compressed. Every sentence earns its place.",
    model=MODEL,
)

Because `Runner.run()` is async, `asyncio.gather` runs all three at once: three drafts
in the time of the slowest one, not the sum of all three.

In [ ]:
task = "Write a post about what breaks first when you put an agent in production."

with trace("Parallel drafts"):
    results = await asyncio.gather(
        Runner.run(professional, task),
        Runner.run(witty, task),
        Runner.run(concise, task),
    )

drafts = [result.final_output for result in results]

for draft in drafts:
    print(draft)
    print("-" * 80)

Now add a fourth agent that judges the drafts. The whole workflow sits inside one
`trace()`, so the traces page shows the fan-out and the decision as a single run.

In [ ]:
picker = Agent(
    name="Picker",
    instructions=(
        "You pick the single best LinkedIn post from the options given. "
        "Imagine you are the reader and pick the one you would stop scrolling for. "
        "Reply with the selected post only, with no explanation."
    ),
    model=MODEL,
)

with trace("Draft and pick"):
    results = await asyncio.gather(
        Runner.run(professional, task),
        Runner.run(witty, task),
        Runner.run(concise, task),
    )
    options = "Post options:\n\n" + "\n\nPost:\n\n".join(result.final_output for result in results)
    best = await Runner.run(picker, options)

print(best.final_output)

## 6. Orchestrating by LLM

### 6a. Agents as tools

`agent.as_tool()` wraps an agent so another agent can call it like a function. Control
returns to the caller when the inner agent finishes, which suits a planner or manager
pattern: **A calls B, B answers, A continues**.

Here one editor agent owns the whole job — it decides how many drafts to request, which
one wins, and when to save.

In [ ]:
description = "Use this tool to write a LinkedIn post. In the input, describe the topic to write about."

writer_tools = [
    professional.as_tool(tool_name="post_writer_professional", tool_description=description),
    witty.as_tool(tool_name="post_writer_witty", tool_description=description),
    concise.as_tool(tool_name="post_writer_concise", tool_description=description),
]

editor_instructions = "You are the editor of an AI engineering LinkedIn account. You commission posts and publish the best one."

editor_task = """Follow these steps:

1. Use each of the three post_writer tools to generate a draft about what breaks first
   when you put an agent in production. Do not continue until all three are ready.
2. Review the drafts and choose the single best one.
3. Use your save_draft tool to save that one post, and only that one."""

editor = Agent(
    name="Editor",
    instructions=editor_instructions,
    tools=[*writer_tools, save_draft],
    model=MODEL,
)

with trace("Editor with agents as tools"):
    result = await Runner.run(editor, editor_task)

print(result.final_output)
print(f"\nDrafts saved so far: {len(saved_drafts)}")

### 6b. Handoffs

A handoff passes control across instead of back: **A hands off to B, and B finishes the
job**. The SDK implements handoffs as tool calls under the hood, so the difference is
about who owns the rest of the conversation, not about the mechanism.

Handoffs can be less reliable than agents-as-tools, especially with smaller models —
here `tool_choice="required"` forces the publisher to actually call its tool rather than
just describing what it would do. If the behaviour is inconsistent, iterate on the
prompts or move up a model size. The traces page shows exactly where a run went astray.

In [ ]:
publisher = Agent(
    name="Publisher",
    instructions=(
        "You are given several draft LinkedIn posts. Pick the best one and save it with your tool. "
        "Save exactly one post."
    ),
    model=MODEL,
    tools=[save_draft],
    model_settings=ModelSettings(tool_choice="required"),
)

handoff_task = """Follow these steps:

1. Use each of the three post_writer tools to generate a draft about what breaks first
   when you put an agent in production. Do not continue until all three are ready.
2. Hand off to the publisher to choose and save the best one."""

editor_with_handoff = Agent(
    name="Editor",
    instructions="You commission posts from your writers, then hand the drafts to the publisher.",
    tools=writer_tools,
    handoffs=[publisher],
    model=MODEL,
)

with trace("Editor with handoff"):
    result = await Runner.run(editor_with_handoff, handoff_task)

print(result.final_output)
print(f"\nDrafts saved so far: {len(saved_drafts)}")

## 7. Structured outputs

Set `output_type` to a Pydantic model and `final_output` comes back as that object
instead of a string. The schema is sent to the model, which is constrained to produce
matching JSON, and the SDK parses it for you. The `Field` descriptions are part of the
schema, so they act as instructions for each field.

This is what turns an agent from something you read into something you can branch on.

In [ ]:
class PostReview(BaseModel):
    """Verdict on a drafted LinkedIn post."""

    is_professional: bool = Field(description="Whether the post reads as professional and credible")
    contains_placeholders: bool = Field(description="Whether the post still contains placeholders such as [first_name]")
    hashtag_count: int = Field(description="How many hashtags the post contains")


PostReview.model_json_schema()

In [ ]:
checker = Agent(
    name="Checker",
    instructions="You review draft LinkedIn posts.",
    model=MODEL,
    output_type=PostReview,
)

sample_post = """Hey [first_name],

AI agents are THE most game changing thing ever and you will be left behind if you
don't get on board right now. DM me to find out more.

#ai #agents #future #innovation #disruption"""

result = await Runner.run(checker, sample_post)
review = result.final_output

print(review)
print(f"\nProfessional? {review.is_professional}")

## 8. Guardrails

A guardrail is a check that stops an agent doing something you do not want. The SDK has
input guardrails (run on the first input to the first agent), output guardrails (run on
the final output of the last agent), and tool guardrails. A guardrail returns a
`GuardrailFunctionOutput`; when `tripwire_triggered` is `True` the run raises instead of
returning, so bad output cannot leak downstream.

Note the gotcha: input and output guardrails only fire at the edges of the run. A
guardrail attached to an agent in the middle of a chain never runs.

In [ ]:
@output_guardrail
async def post_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    message: str,
) -> GuardrailFunctionOutput:
    """Trip the wire when a drafted post is unprofessional or still contains placeholders."""
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review}, tripwire_triggered=is_problem)


sloppy_instructions = (
    BRIEF + "\nAddress the reader by name using a [first_name] placeholder, and be as hyped as you possibly can."
)

sloppy_writer = Agent(
    name="Sloppy Writer",
    instructions=sloppy_instructions,
    model=MODEL,
    output_guardrails=[post_guardrail],
)

try:
    result = await Runner.run(sloppy_writer, "Write a post about AI agents.")
    print(result.final_output)
except OutputGuardrailTripwireTriggered as exc:
    print("Guardrail tripped — the output was blocked.\n")
    print(exc.guardrail_result.output.output_info["review"])

### The same check without the framework

Guardrails are a framework feature, not a requirement. The same protection is a
`Runner.run()` call and an `if` — more obvious, easier to test, and it works in any
framework. Use the built-in version when you want the check enforced automatically at
the edge of a run; use plain code when you want the check where you can see it.

In [ ]:
plain_writer = Agent(name="Sloppy Writer", instructions=sloppy_instructions, model=MODEL)

draft = (await Runner.run(plain_writer, "Write a post about AI agents.")).final_output
review = (await Runner.run(checker, draft)).final_output

if review.contains_placeholders or not review.is_professional:
    print("Rejected, not publishing.\n")
    print(review)
else:
    print(draft)

## Where to go next

- **Hosted tools** — `WebSearchTool`, `FileSearchTool`, `CodeInterpreterTool` and friends
  run on OpenAI's side. Convenient, billed per call, and they tie you to OpenAI:
  <https://openai.github.io/openai-agents-python/tools/#hosted-tools>
- **Other providers** — any OpenAI-compatible endpoint works. Point an `AsyncOpenAI`
  client at the base URL and wrap it in `OpenAIChatCompletionsModel`; tracing still
  needs an OpenAI key, or switch it off with `set_tracing_disabled(True)`.
- **MCP** — connect an agent to Model Context Protocol servers for tools you did not
  write: <https://openai.github.io/openai-agents-python/mcp/>
- **Multi-agent patterns** — the trade-offs between code and LLM orchestration:
  <https://openai.github.io/openai-agents-python/multi_agent/>